<a href="https://colab.research.google.com/github/mahaairshad-shu/Hackathon-SERG_SHU_UoM_Nov2026/blob/main/Rename_BlueTrident_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 0: From the Capture.U export to `train/` and `test/`

> **Already have files named `BT_<MOVEMENT>_<SENSOR>.csv`?** Skip this section and upload them in 0.7.
> **Have the raw export from Capture.U?** Run these cells. They rename the files and put them straight into the `train/` and `test/` folders the rest of the tutorial uses. The upload cells in 0.7 will then see the files and skip the upload.

Capture.U exports files with names like:

```
M Hackathon-3_TS-03342_2026-11-18-15-37-31_aligned.csv     ← one file per sensor per recording
M Hackathon-3_2026-11-18-15-37-31_notes.txt               ← one notes file per recording
```

* `M Hackathon-3` is the **recording** (Capture.U numbers them; numbers can have gaps).
* `TS-03342` is the **sensor's serial number**.
* The notes file holds the time the recording was **captured**.

**What these cells do:**
1. Sort the recordings by capture time.
2. Give them the movement names from `RECORDINGS`, **in the order you recorded them**, and send each to `train/` or `test/`.
3. Replace each serial number by the body position (`SENSOR_POSITIONS`), giving names like `BT_SQUAT_LTHIGH.csv`.

The original files are not changed.

## ⚙️ Rename settings

> ✏️ **What to change**
> * `RECORDINGS`: one line per recording, **in the order you recorded them**: `('MOVEMENT_NAME', 'train')` or `('MOVEMENT_NAME', 'test')`. Use capitals and `_` instead of spaces. Recorded the same movement twice? List it twice: once `'train'`, once `'test'`.
> * `COPY_MISSING_TO_TEST`: if a movement has no `'test'` recording, copy its training files into `test/` so the tutorial still runs. The tutorial will tell you these are copies.
> * `SENSOR_POSITIONS`: which sensor serial number was on which body part. The number is on the back of each sensor and in the file names (`TS-03342` → `'3342'`). Any position names and any number of sensors work.
> * `TRAIN_DIR` / `TEST_DIR` must be the **same** as in the tutorial's ⚙️ Settings (`'train'` and `'test'` by default).
>
> Not sure of your serial numbers? Run up to Step 2: it lists every serial number it finds.

In [1]:
# ======================= ⚙️ RENAME SETTINGS =======================

RAW_DIR = 'raw_capture'       # the Capture.U export is uploaded here
TRAIN_DIR = 'train'           # renamed training files go here   (same as in the tutorial Settings)
TEST_DIR = 'test'             # renamed test files go here       (same as in the tutorial Settings)

# One line per recording, in the order they were RECORDED: (movement name, 'train' or 'test')
RECORDINGS = [
    ('STATIC',                    'train'),
    ('R_HIP_ABDUCTION_ADDUCTION', 'train'),
    ('L_HIP_ABDUCTION_ADDUCTION', 'train'),
    ('R_HIP_FLEXION_EXTENSION',   'train'),
    ('L_HIP_FLEXION_EXTENSION',   'train'),
    ('R_JUMP',                    'train'),
    ('L_JUMP',                    'train'),
    ('SQUAT',                     'train'),
    ('TOE_TOUCHES',               'train'),
    ('WALK',                      'train'),
    ('JUMP',                      'train'),
    # e.g. a second squat recorded later for testing:
    # ('SQUAT',                   'test'),
]

COPY_MISSING_TO_TEST = True   # movement without a 'test' recording -> copy its train files to test/

# Sensor serial number -> body position
SENSOR_POSITIONS = {
    '3359': 'SAC',
    '3342': 'LTHIGH',
    '3347': 'LSHANK',
    '3349': 'LFOOT',
    '3350': 'RTHIGH',
    '3360': 'RSHANK',
    '3348': 'RFOOT',
}

## Upload the Capture.U export

Run the cell and choose the **zip file(s)** exported from Capture.U (or select all the `.csv` and `_notes.txt` files). You can upload several exports at once, e.g. one with your training recordings and one with your test recordings. Then list **all** recordings in `RECORDINGS`, in time order.

In [2]:
import os, re, glob, shutil, zipfile
from datetime import datetime
import pandas as pd

def upload_raw_files(folder):
    os.makedirs(folder, exist_ok=True)
    if glob.glob(os.path.join(folder, '**', '*.csv'), recursive=True):
        print(f'"{folder}" already contains files: upload skipped. '
              f'(To start again: shutil.rmtree("{folder}"))')
        return
    try:
        from google.colab import files as colab_files
    except ImportError:
        print(f'Not running in Colab: copy the Capture.U files into "{folder}/" yourself.')
        return
    for name in colab_files.upload():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall(folder)
            os.remove(name)
        else:
            shutil.move(name, os.path.join(folder, name))

upload_raw_files(RAW_DIR)
print(len(glob.glob(os.path.join(RAW_DIR, '**', '*.*'), recursive=True)), 'files in', RAW_DIR)

Saving OneDrive_2026-11-25_Unnamed.zip to OneDrive_2026-11-25_Unnamed.zip
88 files in raw_capture


## Step 1: Put the recordings in order

Each notes file starts with a line like `Captured, 2026-11-18-15-11-05, 00:00:17`. We read the capture time, sort the recordings from first to last, and match them line by line with `RECORDINGS`.

> ✅ **Check the table:** does each recording have the right movement and set? If not, fix `RECORDINGS` and run again.

In [3]:
TIMESTAMP = r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}'       # e.g. 2026-11-18-15-37-31

rows = []
for path in glob.glob(os.path.join(RAW_DIR, '**', '*_notes.txt'), recursive=True):
    match = re.match(r'^(.*)_(' + TIMESTAMP + r')_notes\.txt$', os.path.basename(path))
    if match is None:
        continue
    with open(path) as f:
        parts = [p.strip() for p in f.readline().split(',')]   # ['Captured', '2026-11-18-15-11-05', '00:00:17']
    rows.append({'recording': match.group(1),                    # 'M Hackathon-3'
                 'export_time': match.group(2),                  # links the notes file to its CSV files
                 'captured': datetime.strptime(parts[1], '%Y-%m-%d-%H-%M-%S'),
                 'duration': parts[2] if len(parts) > 2 else ''})

recordings = pd.DataFrame(rows).sort_values('captured').reset_index(drop=True)
print(f'Found {len(recordings)} recordings; RECORDINGS lists {len(RECORDINGS)}.')
if len(recordings) != len(RECORDINGS):
    display(recordings)
    raise ValueError('The number of recordings and the number of lines in RECORDINGS differ. '
                     'Fix RECORDINGS (or remove extra recordings) and run again.')

recordings['movement'] = [name.upper() for name, _ in RECORDINGS]
recordings['set'] = [s.lower() for _, s in RECORDINGS]
bad_sets = set(recordings['set']) - {'train', 'test'}
if bad_sets:
    raise ValueError(f"Each recording must be 'train' or 'test', not {bad_sets}")
recordings.index = recordings.index + 1
recordings[['recording', 'captured', 'duration', 'movement', 'set']]

Found 11 recordings; RECORDINGS lists 11.


,recording,captured,duration,movement,set
1,M Hackathon-1,2026-11-18 15:09:35,00:00:16,STATIC,train
2,M Hackathon-2,2026-11-18 15:10:25,00:00:18,R_HIP_ABDUCTION_ADDUCTION,train
3,M Hackathon-3,2026-11-18 15:11:05,00:00:17,L_HIP_ABDUCTION_ADDUCTION,train
4,M Hackathon-4,2026-11-18 15:11:55,00:00:19,R_HIP_FLEXION_EXTENSION,train
5,M Hackathon-5,2026-11-18 15:12:25,00:00:18,L_HIP_FLEXION_EXTENSION,train
6,M Hackathon-6,2026-11-18 15:13:44,00:00:16,R_JUMP,train
7,M Hackathon-8,2026-11-18 15:15:21,00:00:18,L_JUMP,train
8,M Hackathon-9,2026-11-18 15:16:03,00:00:19,SQUAT,train
9,M Hackathon-10,2026-11-18 15:16:33,00:00:19,TOE_TOUCHES,train
10,M Hackathon-11,2026-11-18 15:17:05,00:00:25,WALK,train


## Step 2: Match sensor serial numbers to body positions

Every CSV file name contains the recording (`M Hackathon-3`) and the sensor serial number (`TS-03342`). The table lists every serial number found and the position it gets from `SENSOR_POSITIONS`.

> ✅ **Check:** every serial number needs a position. If one shows `❌ not in SENSOR_POSITIONS`, add it to the settings and run again.

In [4]:
def clean_id(sensor_id):
    # 'TS-03342', '03342' and '3342' all become '3342'
    return str(sensor_id).upper().replace('TS-', '').lstrip('0')

positions = {clean_id(k): v.upper() for k, v in SENSOR_POSITIONS.items()}

rows = []
for path in glob.glob(os.path.join(RAW_DIR, '**', '*.csv'), recursive=True):
    match = re.match(r'^(.*)_TS-(\w+?)_(' + TIMESTAMP + ')', os.path.basename(path))
    if match is None:
        print('Skipping (not a Capture.U sensor file):', os.path.basename(path))
        continue
    serial = clean_id(match.group(2))
    rows.append({'recording': match.group(1), 'export_time': match.group(3), 'serial': serial,
                 'position': positions.get(serial, '❌ not in SENSOR_POSITIONS'), 'path': path})
sensor_files = pd.DataFrame(rows)

serials = sensor_files.groupby('serial')['position'].first().to_frame()
serials['files'] = sensor_files.groupby('serial').size()
display(serials)

unknown = sorted(sensor_files.loc[~sensor_files['serial'].isin(positions), 'serial'].unique())
unused = [s for s in positions if s not in set(sensor_files['serial'])]
if unused:
    print('⚠️ In SENSOR_POSITIONS but not found in any file:', unused)
if unknown:
    raise ValueError(f'Serial numbers without a position: {unknown}. Add them to SENSOR_POSITIONS.')

,position,files
serial,,
3342,LTHIGH,11
3347,LSHANK,11
3348,RFOOT,11
3349,LFOOT,11
3350,RTHIGH,11
3359,SAC,11
3360,RSHANK,11


## Step 3: Write the renamed files into `train/` and `test/`

Each sensor file is copied to `TRAIN_DIR` or `TEST_DIR` as `BT_<MOVEMENT>_<POSITION>.csv`, exactly the layout the tutorial expects (see the folder diagram in 0.4).

* `TRAIN_DIR` and `TEST_DIR` are **emptied first**, so running this again does not leave old files behind.
* A movement recorded more than once in the same set gets one sub-folder per take (`take1/`, `take2/`); the tutorial treats each sub-folder as a separate recording.
* With `COPY_MISSING_TO_TEST = True`, movements without a test recording are copied from `train/`.

In [5]:
files = sensor_files.merge(recordings[['recording', 'export_time', 'movement', 'set']],
                           on=['recording', 'export_time'], how='left')
if files['movement'].isna().any():
    missing = sorted(files.loc[files['movement'].isna(), 'recording'].unique())
    raise ValueError(f'No notes file found for recordings: {missing}')

# Number the takes when a movement appears more than once in the same set
recordings['take'] = recordings.groupby(['movement', 'set']).cumcount() + 1
recordings['n_takes'] = recordings.groupby(['movement', 'set'])['movement'].transform('size')
files = files.merge(recordings[['recording', 'export_time', 'take', 'n_takes']], on=['recording', 'export_time'])

for folder in (TRAIN_DIR, TEST_DIR):
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)

def write(row, set_name, note=''):
    folder = TRAIN_DIR if set_name == 'train' else TEST_DIR
    if row['n_takes'] > 1:
        folder = os.path.join(folder, f"take{row['take']}")
    os.makedirs(folder, exist_ok=True)
    new_name = f"BT_{row['movement']}_{row['position']}.csv"
    shutil.copy2(row['path'], os.path.join(folder, new_name))
    return {'set': set_name, 'movement': row['movement'], 'position': row['position'],
            'file': os.path.join(folder, new_name), 'note': note}

log = [write(row, row['set']) for _, row in files.iterrows()]

if COPY_MISSING_TO_TEST:
    tested = set(files.loc[files['set'] == 'test', 'movement'])
    for _, row in files[(files['set'] == 'train') & ~files['movement'].isin(tested)].iterrows():
        log.append(write(row, 'test', note='copy of train'))

log = pd.DataFrame(log)
summary = log.groupby(['set', 'movement']).agg(files=('file', 'count'), note=('note', 'first'))
display(summary)
print(f"{(log['set'] == 'train').sum()} files in '{TRAIN_DIR}/',  {(log['set'] == 'test').sum()} files in '{TEST_DIR}/'")
print('Next: continue with the tutorial. Its upload cells (0.7) will find these files and skip the upload.')

files           note
set   movement                                       
test  JUMP                           7  copy of train
      L_HIP_ABDUCTION_ADDUCTION      7  copy of train
      L_HIP_FLEXION_EXTENSION        7  copy of train
      L_JUMP                         7  copy of train
      R_HIP_ABDUCTION_ADDUCTION      7  copy of train
      R_HIP_FLEXION_EXTENSION        7  copy of train
      R_JUMP                         7  copy of train
      SQUAT                          7  copy of train
      STATIC                         7  copy of train
      TOE_TOUCHES                    7  copy of train
      WALK                           7  copy of train
train JUMP                           7               
      L_HIP_ABDUCTION_ADDUCTION      7               
      L_HIP_FLEXION_EXTENSION        7               
      L_JUMP                         7               
      R_HIP_ABDUCTION_ADDUCTION      7               
      R_HIP_FLEXION_EXTENSION        7               
      R_JUMP                         7               
      SQUAT                          7               
      STATIC                         7               
      TOE_TOUCHES                    7               
      WALK                           7

77 files in 'train/',  77 files in 'test/'
Next: continue with the tutorial. Its upload cells (0.7) will find these files and skip the upload.


## (Optional) Download the organised files

Handy to keep a copy, or to share with your team: this zips `train/` and `test/` into `train.zip` and `test.zip` (the same zips the tutorial asks for) and downloads them.

In [6]:
for folder in (TRAIN_DIR, TEST_DIR):
    zip_path = shutil.make_archive(folder, 'zip', folder)
    print('Created', zip_path)
    try:
        from google.colab import files as colab_files
        colab_files.download(zip_path)
    except ImportError:
        pass

Created /content/train.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Created /content/test.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>